# Quadrotor with Cable-Suspended Payload

Derives the equations of motion for a quadrotor carrying a point-mass payload
suspended by a rigid cable of length $l$.

**Configuration manifold**: $\mathbb{R}^3 \times S^2 \times SO(3)$

| Variable | Space | Description |
|----------|-------|-------------|
| $x$ | $\mathbb{R}^3$ | Quadrotor position |
| $q$ | $S^2$ | Cable direction (unit vector) |
| $R$ | $SO(3)$ | Quadrotor attitude |

**Payload position**: $x_L = x - l q$ (payload hangs below quadrotor)

**Inputs**: Thrust $f$ along body $e_3$, moment $M$ in body frame

Reference: Sreenath, Michael, Kumar (2013)

In [9]:
# Install from GitHub (for Colab); uncomment if not installed locally
# !pip install -q git+https://github.com/vkotaru/pygeomech.git@geomech

from geomech import (
    S2, SO3, Scalar, Vector, Matrix, Dot, Cross,
    SystemVariables, TimeDerivative, Variation,
    compute_eom, to_standard_form, getScalars,
    to_latex, display_eom, display_standard_form,
)
from geomech.core.operations.multiplication import MVMul, SVMul
from geomech.utils.printing import print_tree
from IPython.display import Math

## 1. System definition

In [10]:
# Parameters
mQ = Scalar('m_Q', attr=['Constant'])   # quadrotor mass
mL = Scalar('m_L', attr=['Constant'])   # payload mass
g = Scalar('g', attr=['Constant'])      # gravity
l = Scalar('l', attr=['Constant'])      # cable length
J = Matrix('J', attr=['Constant', 'SymmetricMatrix'])  # quad inertia
e3 = Vector('e3', attr=['Constant'])    # gravity direction
half = Scalar('0.5', value=0.5, attr=['Constant'])

# Configuration variables
x = Vector('x')       # quad position (R3)
q = S2('q')           # cable direction (S2)
R = SO3('R')          # quad attitude (SO3)

Om = R.get_tangent_vector()     # body angular velocity
eta = R.get_variation_vector()  # SO3 variation
omega = q.get_tangent_vector()  # cable angular velocity
xi = q.get_variation_vector()   # S2 variation

# Inputs
f_thrust = Scalar('f')   # thrust magnitude
M_torque = Vector('M')   # body torque

print('x ∈ R³     (quad position)')
print('q ∈ S²     (cable direction)')
print('R ∈ SO(3)  (quad attitude)')

x ∈ R³     (quad position)
q ∈ S²     (cable direction)
R ∈ SO(3)  (quad attitude)


## 2. Kinematics

The payload position is:
$$x_L = x - l q$$

Velocities:
$$\dot{x}_L = \dot{x} - l (\omega \times q)$$

In [11]:
xL = x - l * q

vQ = x.t_diff()   # quad velocity
vL = xL.t_diff()  # payload velocity

display(Math(r'x_L = ' + to_latex(xL)))
display(Math(r'\dot{x}_L = ' + to_latex(vL)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Lagrangian

$$L = \frac{1}{2} m_Q \|\dot{x}\|^2 + \frac{1}{2} m_L \|\dot{x}_L\|^2 + \frac{1}{2} \Omega^T J \Omega - m_Q g\, x \cdot e_3 - m_L g\, x_L \cdot e_3$$

In [12]:
KE = (mQ * Dot(vQ, vQ) * half
      + mL * Dot(vL, vL) * half
      + Dot(Om, J * Om) * half)
PE = mQ * g * Dot(x, e3) + mL * g * Dot(xL, e3)
L = KE - PE

display(Math(r'KE = ' + to_latex(KE)))
display(Math(r'PE = ' + to_latex(PE)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Infinitesimal work

Thrust acts on the quadrotor position, torque on the attitude:
$$\delta W = \delta x \cdot (f R e_3) + \eta \cdot M$$

Note: no direct actuation on the cable — the payload is passive.

In [ ]:
thrust = SVMul(MVMul(R, e3), f_thrust)
dW = Dot(x.delta(), thrust) + Dot(eta, M_torque)

display(Math(r'\delta W = ' + to_latex(dW)))

## 5. Equations of motion

Three coupled equations:
1. **Translational** ($\delta x$): quadrotor + payload translational dynamics
2. **Cable** ($\xi_q$): cable swing dynamics on $S^2$
3. **Rotational** ($\eta_R$): quadrotor attitude dynamics on $SO(3)$

In [ ]:
variables = SystemVariables(vectors=[x, q], matrices=[R])
eom = compute_eom(L, dW, variables)

display_eom(eom)

## 6. Standard form

$$M(q) \ddot{q} + f(q, \dot{q}) + G(q)\, u = 0$$

The mass matrix $M$ has cross-coupling between $\ddot{x}$ and $\dot{\omega}$
(cable angular acceleration couples to quadrotor translation).

In [ ]:
sf = to_standard_form(eom, variables, [thrust, M_torque])
display_standard_form(sf)

## 7. Mass matrix structure

The coupled mass matrix shows the interaction between translational and cable dynamics.

In [ ]:
from geomech.utils.printing import _latex_str_key

for key, eq in sf.items():
    display(Math(f'\\textbf{{{_latex_str_key(key)}}}:'))
    for mk, mv in eq.M.items():
        display(Math(f'  M[{_latex_str_key(mk)}] = {to_latex(mv)}'))
    display(Math(f'  f = {to_latex(eq.f)}'))
    for gk, gv in eq.G.items():
        display(Math(f'  G = {to_latex(gv)}'))
    print()